# SurfaceTarget-CRC — phase 05 — patch 5 1

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.1
#
# CROSS-OMICS EVIDENCE INTEGRATION AND VALIDATION
#
# OBJECTIVES:
# 1. Load completed Phase 2-4 evidence
# 2. Validate all candidate identifiers
# 3. Preserve the original 948-gene research universe
# 4. Validate CPTAC support across previous analyses
# 5. Integrate HGNC-verified DepMap results
# 6. Audit evidence coverage and missingness
# 7. Preserve normal-tissue uncertainty
# 8. Review leading candidates, including TDGF1
# 9. Save canonical cross-omics evidence table
# 10. Save data dictionary and provenance
# 11. Create a new research checkpoint
#
# IMPORTANT:
# - No target ranking
# - No arbitrary scoring weights
# - No automatic candidate exclusions
# - Missing evidence is not negative evidence
# - No cross-study patient-level merging
# - No previous research results overwritten
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np

import hashlib
import json
import re


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RESULTS = ROOT / "results/tables"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [META, RESULTS, CHECKPOINTS]:
    folder.mkdir(parents=True, exist_ok=True)


PHASE3_PATH = (
    RESULTS /
    "surfacetarget_normal_tissue_integration.csv"
)

CPTAC_PATH = (
    RESULTS /
    "cptac_mss_extended_transcriptomic_integration.csv"
)

DEPMAP_PATH = (
    RESULTS /
    "depmap_26q3_hgnc_integrated_dependency_summary.csv"
)

CROSSWALK_PATH = (
    RESULTS /
    "depmap_26q3_hgnc_crosswalk_audit.csv"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.1")
print("CROSS-OMICS EVIDENCE INTEGRATION")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE INPUT FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUT FILES")
print("-" * 65)


required_files = {
    "Phase 3 integration": PHASE3_PATH,
    "CPTAC integration": CPTAC_PATH,
    "DepMap integration": DEPMAP_PATH,
    "HGNC crosswalk": CROSSWALK_PATH
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} missing: {path}"
        )

    print(
        f"{name}: FOUND"
    )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source_hashes = {

    name: sha256_file(path)

    for name, path in required_files.items()

}


print("\nSource files: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD RESEARCH TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING COMPLETED RESEARCH EVIDENCE")
print("-" * 65)


phase3 = pd.read_csv(
    PHASE3_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


cptac = pd.read_csv(
    CPTAC_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


depmap = pd.read_csv(
    DEPMAP_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


crosswalk = pd.read_csv(
    CROSSWALK_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


tables = {
    "Phase3": phase3,
    "CPTAC": cptac,
    "DepMap": depmap,
    "HGNC": crosswalk
}


for name, table in tables.items():

    print(
        f"{name}: {table.shape[0]} genes, "
        f"{table.shape[1]} columns"
    )


# ------------------------------------------------------------
# 4. VALIDATE GENE IDENTITIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-SOURCE GENE IDENTIFIER VALIDATION")
print("-" * 65)


for name, table in tables.items():

    if len(table) != 948:

        raise RuntimeError(
            f"{name}: expected 948 genes."
        )

    if "gene_id" not in table.columns:

        raise RuntimeError(
            f"{name}: gene_id column missing."
        )

    if table["gene_id"].isna().any():

        raise RuntimeError(
            f"{name}: missing gene identifiers."
        )

    if table["gene_id"].duplicated().any():

        raise RuntimeError(
            f"{name}: duplicate gene identifiers."
        )


reference_ids = set(
    phase3["gene_id"]
)


for name, table in tables.items():

    if set(table["gene_id"]) != reference_ids:

        raise RuntimeError(
            f"Candidate gene inventory differs "
            f"between Phase 3 and {name}."
        )


# Verify stable Ensembl IDs wherever available.

for name, table in tables.items():

    if "ensembl_stable_id" not in table.columns:

        continue

    expected = (
        table["gene_id"]
        .astype(str)
        .str.replace(
            r"\.\d+$",
            "",
            regex=True
        )
    )

    observed = (
        table["ensembl_stable_id"]
        .astype(str)
        .str.strip()
    )

    if not expected.eq(observed).all():

        raise RuntimeError(
            f"{name}: stable Ensembl ID mismatch."
        )


print("\nAll four candidate universes agree.")

print("Unique research genes:", len(reference_ids))

print("Ensembl identifier consistency: PASSED")


# ------------------------------------------------------------
# 5. VALIDATE ORIGINAL GENE SYMBOL CORRESPONDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENE SYMBOL CORRESPONDENCE AUDIT")
print("-" * 65)


for name, table in tables.items():

    if "gene_name" not in table.columns:

        continue

    comparison = phase3[
        ["gene_id", "gene_name"]
    ].merge(

        table[
            ["gene_id", "gene_name"]
        ],

        on="gene_id",

        how="inner",

        suffixes=(
            "_phase3",
            "_comparison"
        ),

        validate="one_to_one"
    )

    if name == "Phase3":

        continue

    differences = comparison[
        comparison["gene_name_phase3"]
        .astype(str)
        .ne(
            comparison["gene_name_comparison"]
            .astype(str)
        )
    ]

    print(
        f"{name} symbol differences:",
        len(differences)
    )

    if not differences.empty:

        raise RuntimeError(
            f"{name}: unexpected changes to "
            "original research gene symbols."
        )


print("\nOriginal TCGA gene symbols: PRESERVED")

print(
    "Updated HGNC and DepMap symbols "
    "remain separate annotations."
)


# ------------------------------------------------------------
# 6. CROSS-VALIDATE CPTAC PROTEIN SUPPORT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PROTEOMICS EVIDENCE")
print("-" * 65)


if (
    "protein_enriched" not in phase3.columns
    or "protein_enriched" not in cptac.columns
):

    raise RuntimeError(
        "Protein enrichment evidence missing."
    )


def to_boolean(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


phase3_protein = (
    phase3[
        ["gene_id", "protein_enriched"]
    ]
    .copy()
)


cptac_protein = (
    cptac[
        ["gene_id", "protein_enriched"]
    ]
    .copy()
)


phase3_protein["protein_enriched"] = (
    to_boolean(
        phase3_protein["protein_enriched"]
    )
)


cptac_protein["protein_enriched"] = (
    to_boolean(
        cptac_protein["protein_enriched"]
    )
)


protein_check = phase3_protein.merge(

    cptac_protein,

    on="gene_id",

    validate="one_to_one",

    suffixes=(
        "_phase3",
        "_cptac"
    )
)


if not (
    protein_check[
        "protein_enriched_phase3"
    ]
    ==
    protein_check[
        "protein_enriched_cptac"
    ]
).all():

    raise RuntimeError(
        "CPTAC enrichment status differs "
        "between Phase 2 and Phase 3."
    )


supported_genes = int(
    phase3_protein[
        "protein_enriched"
    ].sum()
)


if supported_genes != 20:

    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


print("\nCPTAC status correspondence: PASSED")

print(
    "Candidates meeting exploratory "
    "cross-omics enrichment criteria:",
    supported_genes
)


print(
    "\nImportant: other genes may have "
    "insufficient proteomic coverage."
)


# ------------------------------------------------------------
# 7. VALIDATE FINAL DEPMAP INTEGRATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING FINAL DEPMAP COVERAGE")
print("-" * 65)


required_depmap_columns = {
    "gene_id",
    "depmap_final_mapping_available",
    "crc_models_measured",
    "crc_median_gene_effect",
    "hgnc_identity_status"
}


if not required_depmap_columns.issubset(
    depmap.columns
):

    raise RuntimeError(
        "Final DepMap integration schema differs."
    )


depmap["depmap_final_mapping_available"] = (
    to_boolean(
        depmap[
            "depmap_final_mapping_available"
        ]
    )
)


mapped_count = int(
    depmap[
        "depmap_final_mapping_available"
    ].sum()
)


if mapped_count != 943:

    raise RuntimeError(
        "Expected 943 verified DepMap mappings."
    )


depmap["crc_models_measured"] = (
    pd.to_numeric(
        depmap["crc_models_measured"],
        errors="raise"
    )
)


measured_count = int(
    depmap[
        "crc_models_measured"
    ].gt(0).sum()
)


if measured_count != 943:

    raise RuntimeError(
        "Expected 943 genes with CRC measurements."
    )


high_coverage_count = int(
    depmap[
        "crc_models_measured"
    ].ge(65).sum()
)


if high_coverage_count != 923:

    raise RuntimeError(
        "Expected 923 high-coverage genes."
    )


print("\nVerified DepMap gene mappings:", mapped_count)

print("Genes with CRC measurements:", measured_count)

print(
    "Genes with >=80% CRC coverage:",
    high_coverage_count
)

print("\nDepMap evidence: VALIDATED")


# ------------------------------------------------------------
# 8. PREPARE DEPMAP EVIDENCE FIELDS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING FUNCTIONAL EVIDENCE FIELDS")
print("-" * 65)


depmap_fields = [
    "depmap_column",
    "depmap_entrez_id",
    "depmap_current_symbol",
    "hgnc_identity_status",
    "depmap_final_mapping_available",
    "crc_models_measured",
    "crc_measurement_fraction",
    "crc_median_gene_effect",
    "crc_q25_gene_effect",
    "crc_q75_gene_effect",
    "crc_fraction_below_minus_05",
    "crc_fraction_below_minus_1",
    "other_models_measured",
    "other_median_gene_effect",
    "crc_minus_other_median_effect",
    "deduplicated_models_measured",
    "deduplicated_median_gene_effect",
    "dedup_minus_primary_median",
    "high_crc_coverage"
]


available_depmap_fields = [
    column
    for column in depmap_fields
    if column in depmap.columns
]


missing_depmap_fields = (
    set(depmap_fields)
    - set(available_depmap_fields)
)


if missing_depmap_fields:

    raise RuntimeError(
        "Expected DepMap evidence missing: "
        f"{missing_depmap_fields}"
    )


depmap_subset = depmap[
    ["gene_id"] + available_depmap_fields
].copy()


# Prefix functional measurements to keep
# their source unambiguous.
#
# Fields already beginning with depmap_
# retain their names.


rename_map = {

    column: (
        column
        if column.startswith("depmap_")
        else f"depmap_{column}"
    )

    for column in available_depmap_fields

}


depmap_subset = depmap_subset.rename(
    columns=rename_map
)


if depmap_subset.columns.duplicated().any():

    raise RuntimeError(
        "Duplicate names in prepared DepMap evidence."
    )


overlap = (
    set(depmap_subset.columns)
    & set(phase3.columns)
) - {"gene_id"}


if overlap:

    raise RuntimeError(
        "Overlapping evidence fields require "
        f"review before integration: {overlap}"
    )


print("\nDepMap evidence fields prepared:",
      len(available_depmap_fields))


# ------------------------------------------------------------
# 9. CONSTRUCT CANONICAL CROSS-OMICS TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING CANONICAL EVIDENCE TABLE")
print("-" * 65)


integrated = phase3.merge(

    depmap_subset,

    on="gene_id",

    how="left",

    validate="one_to_one",

    indicator=True

)


if not integrated["_merge"].eq("both").all():

    raise RuntimeError(
        "Missing gene-level DepMap reconciliation."
    )


integrated = integrated.drop(
    columns="_merge"
)


if len(integrated) != 948:

    raise RuntimeError(
        "Candidate count changed during integration."
    )


if integrated["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes after integration."
    )


integrated["protein_enriched"] = (
    to_boolean(
        integrated["protein_enriched"]
    )
)


print("\nCross-omics integration: PASSED")

print("Candidate genes preserved:", len(integrated))

print("Integrated columns:", len(integrated.columns))


# ------------------------------------------------------------
# 10. PRESERVE NORMAL-TISSUE UNCERTAINTY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMAL-TISSUE EVIDENCE VALIDATION")
print("-" * 65)


if "ihc_record_available" not in integrated.columns:

    raise RuntimeError(
        "Normal-IHC availability indicator missing."
    )


ihc_available = to_boolean(
    integrated["ihc_record_available"]
)


integrated[
    "phase5_normal_ihc_available"
] = ihc_available


ihc_coverage = int(
    ihc_available.sum()
)


if ihc_coverage != 690:

    raise RuntimeError(
        "Expected 690 normal-IHC annotated genes."
    )


# Preserve a separate missing-aware value.
#
# A gene without HPA IHC records should
# not be treated as having zero normal
# tissues with detectable expression.


for source_column in [
    "hpa_tissues_medium_high",
    "hpa_tissues_high",
    "hpa_reliable_medium_high_tissues"
]:

    if source_column not in integrated.columns:

        raise RuntimeError(
            f"Normal-tissue evidence missing: "
            f"{source_column}"
        )

    target_column = (
        "phase5_"
        + source_column
        + "_missing_aware"
    )

    values = pd.to_numeric(
        integrated[source_column],
        errors="raise"
    )

    integrated[target_column] = (
        values.where(
            ihc_available,
            np.nan
        )
    )


print("\nGenes with HPA normal-IHC:", ihc_coverage)

print(
    "Genes without HPA normal-IHC:",
    len(integrated) - ihc_coverage
)


print(
    "Missing normal-tissue evidence "
    "preserved as missing."
)


# ------------------------------------------------------------
# 11. EVIDENCE COVERAGE SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-OMICS EVIDENCE COVERAGE")
print("-" * 65)


def count_true(column):

    if column not in integrated.columns:
        return None

    return int(
        to_boolean(
            integrated[column]
        ).sum()
    )


coverage_records = [

    {
        "source": "TCGA",
        "evidence": "Research screening candidates",
        "genes": 948,
        "interpretation": (
            "Genes retained after the "
            "exploratory transcriptomic workflow"
        )
    },

    {
        "source": "CPTAC",
        "evidence": (
            "Qualifying tumor-associated "
            "protein enrichment"
        ),
        "genes": supported_genes,
        "interpretation": (
            "Exploratory enrichment criteria met; "
            "not therapeutic validation"
        )
    },

    {
        "source": "HPA",
        "evidence": "Normal-tissue IHC records",
        "genes": ihc_coverage,
        "interpretation": (
            "Normal-tissue observations available; "
            "not clinical safety assessment"
        )
    },

    {
        "source": "UniProt",
        "evidence": "Reviewed protein mappings",
        "genes": count_true(
            "uniprot_reviewed_available"
        ),
        "interpretation": (
            "Reviewed protein annotation available"
        )
    },

    {
        "source": "HPA",
        "evidence": "Subcellular annotations",
        "genes": count_true(
            "hpa_subcellular_record_available"
        ),
        "interpretation": (
            "HPA immunofluorescence annotation "
            "available"
        )
    },

    {
        "source": "DepMap",
        "evidence": "Verified gene mappings",
        "genes": mapped_count,
        "interpretation": (
            "HGNC-validated gene identity"
        )
    },

    {
        "source": "DepMap",
        "evidence": "High-coverage CRC gene effects",
        "genes": high_coverage_count,
        "interpretation": (
            "Measurements in at least "
            "65 of 81 CRC models"
        )
    }

]


coverage = pd.DataFrame(
    coverage_records
)


print("\n")

print(
    coverage[
        ["source", "evidence", "genes"]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 12. AUDIT AVAILABLE EVIDENCE COLUMNS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING CANDIDATE EVIDENCE SCHEMA")
print("-" * 65)


rna_keywords = (
    r"log2|fold|padj|tier_a|"
    r"paired|expression|tumor"
)


protein_keywords = (
    r"protein|proteomic|cptac|"
    r"wilcoxon|mss_complete"
)


rna_columns = [
    column
    for column in phase3.columns
    if re.search(
        rna_keywords,
        column,
        flags=re.IGNORECASE
    )
]


protein_columns = [
    column
    for column in phase3.columns
    if re.search(
        protein_keywords,
        column,
        flags=re.IGNORECASE
    )
]


print("\nPotential RNA evidence fields:")

for column in rna_columns[:35]:
    print(" ", column)


print("\nPotential protein evidence fields:")

for column in protein_columns[:35]:
    print(" ", column)


print(
    "\nFull field inventory will be saved."
)


# ------------------------------------------------------------
# 13. REVIEW LEADING CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CROSS-OMICS CANDIDATE REVIEW")
print("-" * 65)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


leading = integrated[
    integrated["gene_name"]
    .isin(LEADING_GENES)
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


leading_columns = [
    "gene_name",
    "protein_enriched",
    "uniprot_topology_category",
    "uniprot_gpi_anchor",
    "phase5_normal_ihc_available",
    "phase5_hpa_tissues_medium_high_missing_aware",
    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect",
    "depmap_crc_fraction_below_minus_05"
]


missing_leading_columns = (
    set(leading_columns)
    - set(integrated.columns)
)


if missing_leading_columns:

    raise RuntimeError(
        "Leading evidence columns missing: "
        f"{missing_leading_columns}"
    )


print("\n")

print(
    leading[
        leading_columns
    ]
    .to_string(
        index=False
    )
)


print("\nIMPORTANT:")

print(
    "Protein enrichment, membrane topology, "
    "normal-tissue expression and CRISPR "
    "dependency are distinct evidence types."
)


print(
    "No biological target rankings "
    "have been assigned."
)


# ------------------------------------------------------------
# 14. FINAL INTEGRITY CHECKS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FINAL CROSS-OMICS INTEGRITY VALIDATION")
print("-" * 65)


if int(
    integrated["protein_enriched"].sum()
) != 20:

    raise RuntimeError(
        "Protein support count changed."
    )


if int(
    to_boolean(
        integrated[
            "depmap_final_mapping_available"
        ]
    ).sum()
) != 943:

    raise RuntimeError(
        "DepMap mapping count changed."
    )


if int(
    integrated[
        "phase5_normal_ihc_available"
    ].sum()
) != 690:

    raise RuntimeError(
        "Normal-IHC coverage changed."
    )


unmapped = integrated[
    ~to_boolean(
        integrated[
            "depmap_final_mapping_available"
        ]
    )
].copy()


if len(unmapped) != 5:

    raise RuntimeError(
        "Expected five candidates without "
        "DepMap mappings."
    )


if pd.to_numeric(
    unmapped[
        "depmap_crc_models_measured"
    ],
    errors="coerce"
).fillna(0).gt(0).any():

    raise RuntimeError(
        "Unmapped genes unexpectedly have "
        "CRISPR measurements."
    )


print("\nCandidate preservation: PASSED")

print("Gene identifier correspondence: PASSED")

print("CPTAC evidence consistency: PASSED")

print("DepMap measurement coverage: PASSED")

print("Normal-IHC missingness handling: PASSED")


# ------------------------------------------------------------
# 15. CREATE EVIDENCE FIELD DICTIONARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING EVIDENCE FIELD DICTIONARY")
print("-" * 65)


field_records = []


for column in integrated.columns:

    if column.startswith("depmap_"):

        source = "DepMap"

    elif column.startswith("phase5_"):

        source = "Phase 5 derived audit field"

    else:

        source = (
            "Phase 3 integrated evidence "
            "(TCGA/CPTAC/HPA/UniProt)"
        )


    field_records.append({

        "column_name": column,

        "source": source,

        "dtype": str(
            integrated[column].dtype
        ),

        "nonmissing_records": int(
            integrated[column]
            .notna()
            .sum()
        ),

        "missing_records": int(
            integrated[column]
            .isna()
            .sum()
        )

    })


field_dictionary = pd.DataFrame(
    field_records
)


print(
    "\nFields documented:",
    len(field_dictionary)
)


# ------------------------------------------------------------
# 16. SAVE CANONICAL EVIDENCE TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CANONICAL CROSS-OMICS RESULTS")
print("-" * 65)


OUTPUT_PATH = (
    RESULTS /
    "surfacetarget_phase5_integrated_evidence_v1.csv"
)


integrated.to_csv(
    OUTPUT_PATH,
    index=False
)


coverage.to_csv(
    RESULTS /
    "surfacetarget_phase5_data_coverage.csv",
    index=False
)


field_dictionary.to_csv(
    RESULTS /
    "surfacetarget_phase5_field_dictionary.csv",
    index=False
)


leading.to_csv(
    RESULTS /
    "surfacetarget_phase5_leading_gene_audit.csv",
    index=False
)


unmapped.to_csv(
    RESULTS /
    "surfacetarget_phase5_unmeasured_depmap_genes.csv",
    index=False
)


# Verify that the complete canonical table
# can be reopened without losing gene rows.


reloaded = pd.read_csv(
    OUTPUT_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


if len(reloaded) != 948:

    raise RuntimeError(
        "Saved integrated table row count differs."
    )


if (
    reloaded["gene_id"].tolist()
    != integrated["gene_id"].tolist()
):

    raise RuntimeError(
        "Saved gene ordering or identifiers differ."
    )


print("\nCanonical evidence table saved.")

print("Saved table verification: PASSED")


# ------------------------------------------------------------
# 17. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PHASE 5 PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_files": {

        name: str(path)

        for name, path in required_files.items()

    },

    "source_sha256": source_hashes,

    "candidate_genes": 948,

    "cross_omics_enriched_candidates": 20,

    "normal_ihc_annotated_candidates": 690,

    "depmap_verified_mappings": 943,

    "depmap_high_coverage_genes": 923,

    "canonical_output": str(OUTPUT_PATH),

    "canonical_output_sha256":
        sha256_file(OUTPUT_PATH),

    "integration_key": "gene_id",

    "integration_policy": (
        "One-to-one integration using the "
        "original versioned TCGA Ensembl ID"
    ),

    "source_separation": (
        "TCGA, CPTAC and DepMap observations "
        "remain independent datasets"
    ),

    "normal_ihc_missingness": (
        "Missing HPA IHC records are retained "
        "as unavailable evidence"
    ),

    "protein_missingness": (
        "Failure to meet CPTAC enrichment "
        "criteria does not imply absent protein"
    ),

    "msi_limitations": (
        "TCGA discovery Tier B is computationally "
        "low-MSI, not independently confirmed MSS; "
        "DepMap CRC models are not MSS-confirmed"
    ),

    "target_ranking_performed": False,

    "candidate_exclusions": 0,

    "limitations": [

        "Transcriptomic differential expression "
        "and CPTAC enrichment originate from "
        "different patient cohorts.",

        "Bulk tumor measurements cannot "
        "establish malignant-cell-specific expression.",

        "HPA staining is descriptive "
        "and not a clinical safety measurement.",

        "UniProt topology does not "
        "demonstrate antibody accessibility.",

        "DepMap effects do not directly "
        "measure therapeutic antibody efficacy.",

        "Missing evidence has not been "
        "converted to negative evidence.",

        "No therapeutic targets "
        "have been experimentally validated."

    ]

}


with open(
    META /
    "surfacetarget_phase5_integration_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nIntegration provenance saved.")


# ------------------------------------------------------------
# 18. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch51_{timestamp}.zip"
)


# Includes reproducible processed results
# and compact research sources.
#
# Excludes the large raw TCGA and DepMap
# datasets and preserved upload fragments.


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        source_folder = ROOT / folder

        if not source_folder.exists():
            continue

        for file_path in source_folder.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size
    / 1024**2
)


print("\nCheckpoint created.")

print(
    "Size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Location:")
print(checkpoint)


# ------------------------------------------------------------
# 19. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.1 SUMMARY")
print("=" * 65)


print("\nIntegrated candidate genes:", len(integrated))

print("Integrated evidence columns:", len(integrated.columns))

print("CPTAC-supported candidates:", supported_genes)

print("HPA normal-IHC annotated genes:", ihc_coverage)

print("Verified DepMap mappings:", mapped_count)

print(
    "Genes with high DepMap coverage:",
    high_coverage_count
)


print("\nSaved outputs:")

print("1. surfacetarget_phase5_integrated_evidence_v1.csv")

print("2. surfacetarget_phase5_data_coverage.csv")

print("3. surfacetarget_phase5_field_dictionary.csv")

print("4. surfacetarget_phase5_leading_gene_audit.csv")

print("5. surfacetarget_phase5_unmeasured_depmap_genes.csv")

print("6. surfacetarget_phase5_integration_provenance.json")


print("\nResearch checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nNo candidate genes excluded.")

print("No therapeutic target scores assigned.")

print("No earlier analyses overwritten.")

print("\nPATCH 5.1 COMPLETE")

print("=" * 65)